# panelcast quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cupidthatbtc/panelcast/blob/main/examples/quickstart.ipynb)

Fit a hierarchical Bayesian panel model end-to-end on the bundled aerospace
example — airframes flying scored test flights — and plot its calibrated
prediction intervals. Runs on a plain CPU runtime in a few minutes.

This is a *tiny smoke-scale* fit (1 chain, relaxed convergence gates), meant to
show the pipeline and its artifacts — not publication-grade inference. The
publication configuration and its gates are described in the
[model card](https://github.com/cupidthatbtc/panelcast/blob/main/MODEL_CARD.md).

In [ ]:
# On Colab this installs panelcast; anywhere it is already installed
# (a checkout, CI) the import succeeds and the cell is a no-op. The git
# source keeps the badge working independent of PyPI publication lag.
try:
    import panelcast  # noqa: F401
except ImportError:
    %pip install -q "panelcast @ git+https://github.com/cupidthatbtc/panelcast@main"
    import panelcast  # noqa: F401
print(panelcast.__version__)

In [ ]:
import os
from pathlib import Path

# Colab's default runtime has no GPU JAX; pin the CPU backend explicitly so
# behavior is identical everywhere.
os.environ.setdefault("JAX_PLATFORMS", "cpu")

# The pipeline writes data/, models/, and outputs/ under the current
# directory; work in a scratch dir so nothing else is touched.
workdir = Path("quickstart_workdir").resolve()
workdir.mkdir(exist_ok=True)
os.chdir(workdir)
print(f"working in {workdir}")

## Configure and run the pipeline

Everything below is the public Python API: a `PipelineConfig` pointed at the
bundled aerospace dataset descriptor, run through `run_pipeline`. The
descriptor is one YAML file mapping the domain's column names and score
bounds — retargeting panelcast to your own panel means writing one of these
(see [`docs/PORTING.md`](https://github.com/cupidthatbtc/panelcast/blob/main/docs/PORTING.md)).

In [ ]:
from panelcast.config.descriptor import resolve_demo_descriptor_path
from panelcast.pipelines.orchestrator import PipelineConfig, run_pipeline

descriptor_path = resolve_demo_descriptor_path("aero")

# Tiny, tolerant smoke settings (the same shape `panelcast demo` uses):
# 1 chain x 300 draws, relaxed gates, divergences allowed.
config = PipelineConfig(
    seed=42,
    dataset=str(descriptor_path),
    num_chains=1,
    num_samples=300,
    num_warmup=300,
    max_albums=10,
    min_albums_filter=2,
    rhat_threshold=1.1,
    ess_threshold=100,
    allow_divergences=True,
    strict=False,
    enforce_lockfile=False,
)

exit_code = run_pipeline(config)
assert exit_code == 0, f"pipeline failed with exit code {exit_code}"

## Read the run's metrics

Every run lands in a timestamped directory under `outputs/`, with a manifest,
the resolved configuration, evaluation metrics, and a generated model card.

In [ ]:
import json

from panelcast.paths import resolve_latest

run_dir = resolve_latest()
assert run_dir is not None
metrics = json.loads((run_dir / "evaluation" / "metrics.json").read_text(encoding="utf-8"))
primary_split = metrics.get("primary_split", "within_entity_temporal")
primary = metrics["splits"][primary_split]
point = primary["point_metrics"]
coverages = primary["calibration"]["coverages"]

print(f"run: {run_dir.name}")
print(f"held-out n:  {point['n_observations']}")
print(f"MAE:         {point['mae']:.3f}")
print(f"R^2:         {point['r2']:.3f}")
for level, cov in sorted(coverages.items()):
    print(f"{float(level):.0%} interval -> empirical coverage {cov['empirical']:.1%}")

## Calibrated intervals on held-out flights

The per-row prediction arrays (with interval bounds) are part of the run's
evaluation artifacts — no recomputation needed.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

pred = json.loads(
    (run_dir / "evaluation" / primary_split / "predictions.json").read_text(
        encoding="utf-8"
    )
)
y = np.asarray(pred["y_true"], dtype=float)
mu = np.asarray(pred["y_pred_mean"], dtype=float)
lo = np.asarray(pred["y_pred_lower"], dtype=float)
hi = np.asarray(pred["y_pred_upper"], dtype=float)

order = np.argsort(mu)
x = np.arange(len(order))
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.errorbar(
    x,
    mu[order],
    yerr=[mu[order] - lo[order], hi[order] - mu[order]],
    fmt="o",
    ms=3,
    lw=0.8,
    alpha=0.6,
    label="predicted (interval)",
)
ax.plot(x, y[order], "k.", ms=4, label="actual")
inside = float(np.mean((y >= lo) & (y <= hi)))
ax.set_title(f"Held-out next flights — {inside:.0%} of actuals inside the interval")
ax.set_xlabel("held-out flight (sorted by predicted score)")
ax.set_ylabel("performance score")
ax.legend()
fig.tight_layout()
plt.show()

## Point it at your own panel

1. Put your data in a CSV: one row per event, with an entity column, a date,
   a bounded score, and an observation count.
2. Write a dataset descriptor YAML naming those columns and bounds
   ([`docs/PORTING.md`](https://github.com/cupidthatbtc/panelcast/blob/main/docs/PORTING.md)
   walks through the aerospace one used here).
3. Rerun the cell above with `dataset="path/to/your_descriptor.yaml"` and an
   `AOTY_DATASET_PATH`-style env var — or the descriptor's own `raw_path_default`.

For real inference, raise the sampler to the publication preset (4 chains ×
5000 draws) and keep the convergence, PPC, and calibration gates strict —
that is the whole point of the tool.